# Cloud and local CPU training

This notebook trains four grid/action SAC policies with the shared `GridOptimizeEnv`. The Newton–Raphson power-flow solve is CPU-bound; a GPU is optional.

The notebook uses the same trainer and CSV schema as the local CLI. It writes `rl_optimizer_{shed_load|redispatch_gen}_{39|118}.zip` and matching `_rewards.csv` files under `models/`. Keep training output in a separate directory or pass `overwrite=True` only when replacing an existing checkpoint intentionally.

In Colab, clone or upload the repository first, then set `PROJECT_ROOT` to its root. The root is discovered from the current directory and its parents below.


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "gymnasium==1.3.0", "stable-baselines3==2.9.0", "pandapower==3.4.0"])


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / 'modules' / 'config.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from the Project SWING repository or set PROJECT_ROOT to its root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train_rl_agent import train

MODELS_DIR = PROJECT_ROOT / 'models'

def train_policy(grid, action, timesteps=50_000, seed=0, overwrite=False):
    return train(grid, action, timesteps, MODELS_DIR, seed=seed, overwrite=overwrite)


In [ ]:
# This trains each of the four supported policies sequentially.
# Use a smaller timestep count for a smoke run. Existing artifacts are protected by default.

for grid, action in (("39", "SHED_LOAD"), ("39", "REDISPATCH_GEN"), ("118", "SHED_LOAD"), ("118", "REDISPATCH_GEN")):
    artifact = train_policy(grid, action, timesteps=50_000, seed=0)
    print(f"Saved {artifact}")
